In [0]:
-- ============================================
-- BRONZE 层: 增量数据加载 (使用 MERGE)
-- ============================================

-- 1. 创建临时视图连接 MySQL (使用 Databricks 的 MySQL 连接器)
CREATE OR REPLACE TEMPORARY VIEW mysql_order_info_latest AS
SELECT 
    id                    ,
      consignee             ,
      consignee_tel         ,
      total_amount          ,
      order_status          ,
      user_id               ,
      payment_way           ,
      delivery_address      ,
      order_comment         ,
      out_trade_no          ,
      trade_body            ,
      create_time           ,
      operate_time          ,
      expire_time           ,
      process_status        ,
      tracking_no           ,
      parent_order_id       ,
      img_url               ,
      province_id           ,
      activity_reduce_amount,
      coupon_reduce_amount  ,
      original_total_amount ,
      feight_fee            ,
      feight_fee_reduce     ,
      refundable_time       ,
    CURRENT_TIMESTAMP() as _bronze_load_ts,
    'mysql_production' as _source_system,
    'order_info' as _source_table
FROM awsmysql_catalog.gmall.order_info
WHERE operate_time > (
    SELECT COALESCE(MAX(operate_time), '1900-01-01')
    FROM aws3.bronze.orders_info
);
---所以要定期-- 清理超过保留期限的文件（默认清理7天前的）我的表设置一天
--VACUUM your_table_name;实际数据在云上，但对DB，已经没了


-- 2. MERGE 到 Bronze 层
MERGE INTO aws3.bronze.orders_info AS target
USING mysql_order_info_latest AS source
ON target.id = source.id 
   AND target._source_system = source._source_system
   AND target.operate_time = source.operate_time
WHEN NOT MATCHED THEN
INSERT (
    _bronze_load_ts,
    _bronze_load_id,
    _source_system,
    _source_table,
    id                    ,
    consignee             ,
    consignee_tel         ,
    total_amount          ,
    order_status          ,
    user_id               ,
    payment_way           ,
    delivery_address      ,
    order_comment         ,
    out_trade_no          ,
    trade_body            ,
    create_time           ,
    operate_time          ,
    expire_time           ,
    process_status        ,
    tracking_no           ,
    parent_order_id       ,
    img_url               ,
    province_id           ,
    activity_reduce_amount,
    coupon_reduce_amount  ,
    original_total_amount ,
    feight_fee            ,
    feight_fee_reduce     ,
    refundable_time       
)
VALUES (
    source._bronze_load_ts,
    UUID() ,--as _bronze_load_id,
    source._source_system,
    source._source_table,
    source.id                    ,
    source.consignee             ,
    source.consignee_tel         ,
    source.total_amount          ,
    source.order_status          ,
    source.user_id               ,
    source.payment_way           ,
    source.delivery_address      ,
    source.order_comment         ,
    source.out_trade_no          ,
    source.trade_body            ,
    source.create_time           ,
    source.operate_time          ,
    source.expire_time           ,
    source.process_status        ,
    source.tracking_no           ,
    source.parent_order_id       ,
    source.img_url               ,
    source.province_id           ,
    source.activity_reduce_amount,
    source.coupon_reduce_amount  ,
    source.original_total_amount ,
    source.feight_fee            ,
    source.feight_fee_reduce     ,
    source.refundable_time       
);